**Recovery note:** This notebook was reconstructed from an HTML export. The code, narrative, and visible outputs were preserved where possible.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


<h2 id="Load-datasets">Load datasets<a class="anchor-link" href="#Load-datasets">¶</a></h2><p>Load the AIS position and vessel track datasets, merge the position files, and create working copies for preprocessing.</p>


In [2]:
from pathlib import Path

# Load datasets
DATA_PATH = Path("../data/raw")

positions_part1 = pd.read_csv(DATA_PATH / "vessel_positions_part1.csv")
positions_part2 = pd.read_csv(DATA_PATH / "vessel_positions_part2.csv")
tracks = pd.read_csv(DATA_PATH / "vessel_tracks.csv")

# Merge position datasets
positions = pd.concat(
    [positions_part1, positions_part2],
    ignore_index=True
)

# Create working copies
positions_clean = positions.copy()
tracks_clean = tracks.copy()

print(f"Positions: {positions_clean.shape}")
print(f"Tracks: {tracks_clean.shape}")


Positions: (1534210, 17)
Tracks: (1196, 24)


<h2 id="Inspect-missing-values">Inspect missing values<a class="anchor-link" href="#Inspect-missing-values">¶</a></h2><p>Summarize missing values and data types to identify variables requiring cleaning or special handling during preprocessing.</p>


In [3]:
def missing_summary(df):
    """Return a summary of missing values and data types."""

    missing = (
        df.isna()
        .sum()
        .rename("Missing Values")
        .to_frame()
    )

    missing["Percentage"] = (
        missing["Missing Values"] / len(df) * 100
    ).round(2)

    missing["Data Type"] = df.dtypes.astype(str)

    missing = (
        missing[missing["Missing Values"] > 0]
        .sort_values("Missing Values", ascending=False)
    )

    return missing[["Data Type", "Missing Values", "Percentage"]]


In [4]:
print("Position Missing Values and Data Types")
display(missing_summary(positions_clean))

print("\nTracks Missing Values and Data Types")
display(missing_summary(tracks_clean))


Position Missing Values and Data Types


,Data Type,Missing Values,Percentage
destination,str,794071,51.76
draught,float64,761359,49.63
nav_status,float64,622472,40.57
heading,float64,565136,36.84
cog,float64,563981,36.76
sog,float64,563971,36.76
name,str,146235,9.53



Tracks Missing Values and Data Types


,Data Type,Missing Values,Percentage
destination,str,760,63.55
dimensions,str,746,62.37
eta,str,746,62.37
draught,float64,746,62.37
callsign,str,701,58.61
ship_type,float64,699,58.44
name,str,650,54.35
nav_status,float64,130,10.87
heading,float64,63,5.27
cog,float64,59,4.93


<h1 id="Preprocessing">Preprocessing<a class="anchor-link" href="#Preprocessing">¶</a></h1>


<h2 id="Replace-AIS-placeholder-values">Replace AIS placeholder values<a class="anchor-link" href="#Replace-AIS-placeholder-values">¶</a></h2><p>AIS uses predefined placeholder codes to represent not avaiable  These values are  therefore replaced with <code>NaN</code></p>


In [5]:
positions_clean["heading"] = positions_clean["heading"].replace(511, np.nan)
tracks_clean["heading"] = tracks_clean["heading"].replace(511, np.nan)

positions_clean["cog"] = positions_clean["cog"].replace(360, np.nan)
tracks_clean["cog"] = tracks_clean["cog"].replace(360, np.nan)

positions_clean["sog"] = positions_clean["sog"].replace(1023, np.nan)
tracks_clean["sog"] = tracks_clean["sog"].replace(1023, np.nan)

tracks_clean["eta"] = tracks_clean["eta"].replace("—", np.nan)


In [6]:
print(f"Heading placeholder (511): {(positions_clean['heading'] == 511).sum()}")
print(f"COG placeholder (360): {(positions_clean['cog'] == 360).sum()}")
print(f"SOG placeholder (1023): {(positions_clean['sog'] == 1023).sum()}")
print(f"ETA placeholder (—): {(tracks_clean['eta'] == '—').sum()}")


Heading placeholder (511): 0
COG placeholder (360): 0
SOG placeholder (1023): 0
ETA placeholder (—): 0


<h3 id="Findings">Findings<a class="anchor-link" href="#Findings">¶</a></h3><p>AIS uses predefined placeholder codes to indicate unavailable values. These placeholders are replaced with NaN to distinguish missing information from valid observation</p>


<h2 id="Convert-Data-Types">Convert Data Types<a class="anchor-link" href="#Convert-Data-Types">¶</a></h2><p>Convert timestamp columns to datetime so they can be used in time-based preprocessing.</p>


In [7]:
# Convert date columns to datetime

date_columns = {
    "positions": ["recorded_at", "created_at", "updated_at"],
    "tracks": ["last_update", "created_at", "updated_at"]
}

positions_clean[date_columns["positions"]] = (
    positions_clean[date_columns["positions"]]
    .apply(pd.to_datetime)
)

tracks_clean[date_columns["tracks"]] = (
    tracks_clean[date_columns["tracks"]]
    .apply(pd.to_datetime)
)

print("Positions")
display(positions_clean[date_columns["positions"]].dtypes)

print("\nTracks")
display(tracks_clean[date_columns["tracks"]].dtypes)


Positions


recorded_at    datetime64[us]
created_at     datetime64[us]
updated_at     datetime64[us]
dtype: object



Tracks


last_update    datetime64[us]
created_at     datetime64[us]
updated_at     datetime64[us]
dtype: object


<h2 id="Inspect-ETA">Inspect ETA<a class="anchor-link" href="#Inspect-ETA">¶</a></h2>


In [8]:
print("Missing ETA:", tracks_clean["eta"].isna().sum())
tracks_clean["eta"].dropna().head(3)


Missing ETA: 757


2    04/05 07:00
4    03/25 15:00
5    12/10 08:30
Name: eta, dtype: str


In [9]:
print(
    "Tracks:",
    tracks_clean["last_update"].min(),
    "to",
    tracks_clean["last_update"].max()
)

print(
    "Positions:",
    positions_clean["recorded_at"].min(),
    "to",
    positions_clean["recorded_at"].max()
)


Tracks: 2026-04-03 12:14:09 to 2026-04-16 15:39:14
Positions: 2026-04-03 10:42:38 to 2026-04-16 12:43:53


<p>Both datasets cover April 2026. That is why 2026 is used as the reference year when reconstructing the reported ETA timestamps.</p>


<h2 id="Reconstruct-ETA">Reconstruct ETA<a class="anchor-link" href="#Reconstruct-ETA">¶</a></h2>


In [10]:
reference_year = tracks_clean["last_update"].dt.year.mode().iloc[0]

tracks_clean["eta_datetime"] = pd.to_datetime(
    f"{reference_year}/" + tracks_clean["eta"],
    format="%Y/%m/%d %H:%M",
    errors="coerce"
)

print(f"Reference year: {reference_year}")
print(f"Reconstructed ETA timestamps: {tracks_clean['eta_datetime'].notna().sum():,}")
print(f"Missing ETA timestamps: {tracks_clean['eta_datetime'].isna().sum():,}")


Reference year: 2026
Reconstructed ETA timestamps: 439
Missing ETA timestamps: 757


In [11]:
tracks_clean["eta"].dropna().str[:2].value_counts().sort_index()


eta
01     12
02      1
03     11
04    395
05      9
07      2
08      1
09      2
10      2
11      1
12      3
Name: count, dtype: int64


<h2 id="Compare-track-and-position-timestamps">Compare track and position timestamps<a class="anchor-link" href="#Compare-track-and-position-timestamps">¶</a></h2><h3 id="Does-the-tracks-dataset-represent-the-latest-vessel-state-compared-with-the-position-messages?">Does the tracks dataset represent the latest vessel state compared with the position messages?<a class="anchor-link" href="#Does-the-tracks-dataset-represent-the-latest-vessel-state-compared-with-the-position-messages?">¶</a></h3>


In [12]:
latest_positions = (
    positions_clean
    .groupby("mmsi")["recorded_at"]
    .max()
    .rename("latest_position")
)

comparison = (
    tracks_clean[["mmsi", "last_update"]]
    .merge(latest_positions, on="mmsi", how="left")
)

comparison["difference_minutes"] = (
    comparison["last_update"] - comparison["latest_position"]
).dt.total_seconds() / 60

comparison["difference_minutes"].describe()


count     1178.000000
mean        21.717600
std        370.593366
min        -10.966667
25%        -10.262500
50%         -9.966667
75%         -9.600000
max      10942.133333
Name: difference_minutes, dtype: float64


<p>Most vessels last_update(tracks)  is within approximately 10 minutes of the latest recorded AIS position. Which means that t tracks_clean represents a recent vessel-level snapshot and can be used to provide vessel-level information during feature engineering.</p>
<p><em>last_update - recorded at</em></p>


<h2 id="Validate-reconstructed-ETA-timestamps">Validate reconstructed ETA timestamps<a class="anchor-link" href="#Validate-reconstructed-ETA-timestamps">¶</a></h2><p>The reconstructed ETA timestamps are compared with the vessel's <code>last_update</code>. Since the reported ETA represents a future arrival:</p>
<p>A reported ETA must happen  after the vessel's <code>last_update</code>An ETA before the last update is impossible</p>


In [13]:
# Validate reconstructed ETAs against the vessel's last update
#ETA must be after last update 
valid_eta = (
    tracks_clean["eta_datetime"] >= tracks_clean["last_update"]
)

print(f"Total vessels: {len(tracks_clean):,}")
print(f"Valid ETAs: {valid_eta.sum():,}")
print(f"ETAs before last_update: {(~valid_eta & tracks_clean['eta_datetime'].notna()).sum():,}") #This flips every Boolean:
print(f"Missing ETAs: {tracks_clean['eta_datetime'].isna().sum():,}")
#True  → missing
#False → not missing
#sum() counts the True values.


Total vessels: 1,196
Valid ETAs: 379
ETAs before last_update: 60
Missing ETAs: 757


<h3 id="Retain-chronologically-valid-ETA-labels">Retain chronologically valid ETA labels<a class="anchor-link" href="#Retain-chronologically-valid-ETA-labels">¶</a></h3><p>Keep only vessels with a valid, chronologically consistent ETA.</p>


In [14]:
# Retain chronologically valid vessel-level ETA labels

tracks_valid_eta = tracks_clean.loc[valid_eta].copy()

print(f"Total vessels: {len(tracks_clean):,}")
print(f"Vessels with valid ETAs: {len(tracks_valid_eta):,}")
print(f"Excluded missing ETAs: {tracks_clean['eta_datetime'].isna().sum():,}")
print(f"Excluded chronologically invalid ETAs: {(~valid_eta & tracks_clean['eta_datetime'].notna()).sum():,}")


Total vessels: 1,196
Vessels with valid ETAs: 379
Excluded missing ETAs: 757
Excluded chronologically invalid ETAs: 60


<h3 id="Inspect-the-ETA-horizon">Inspect the ETA horizon<a class="anchor-link" href="#Inspect-the-ETA-horizon">¶</a></h3><p>We want to eventually caluclate &gt;  remaining_hours = ETA - recorded_at</p>
<p>So we  need to check whether some reported ETAs are unrealistically far away.
The time between <code>last_update</code> and the reported ETA is calculated to assess the distribution of the retained ETA labels.</p>
<p>hours_to_eta_at_last_update =
eta_datetime - last_update</p>


In [15]:
tracks_valid_eta["hours_to_eta_at_last_update"] = (
    tracks_valid_eta["eta_datetime"]
    - tracks_valid_eta["last_update"]
).dt.total_seconds() / 3600

summary = tracks_valid_eta["hours_to_eta_at_last_update"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

display(summary)

print("\nETA horizon in days:")
print(f"Median: {summary['50%']/24:.1f} days")
print(f"75th percentile: {summary['75%']/24:.1f} days")
print(f"90th percentile: {summary['90%']/24:.1f} days")
print(f"95th percentile: {summary['95%']/24:.1f} days")
print(f"Maximum: {summary['max']/24:.1f} days")


count     379.000000
mean      206.233977
std       714.188725
min         1.418056
25%        30.237639
50%        56.191944
75%        90.616250
90%       302.201222
95%       503.397750
99%      4290.068094
max      5716.140556
Name: hours_to_eta_at_last_update, dtype: float64



ETA horizon in days:
Median: 2.3 days
75th percentile: 3.8 days
90th percentile: 12.6 days
95th percentile: 21.0 days
Maximum: 238.2 days


<h3 id="Findings">Findings<a class="anchor-link" href="#Findings">¶</a></h3><ul>
<li>Most reported ETA horizons fall within 4 days.</li>
<li>95% of reported ETAs are within approximately 21 days.</li>
<li>Only a small number of records have much longer ETA horizons.</li>
<li>238 days is extremely long</li>
</ul>


<h3 id="Inspect-extreme-ETA-horizons">Inspect extreme ETA horizons<a class="anchor-link" href="#Inspect-extreme-ETA-horizons">¶</a></h3><p>Vessels with reported ETAs more than 90 days after <code>last_update</code> were manually inspected</p>


In [16]:
long_eta = tracks_valid_eta.loc[
    tracks_valid_eta["hours_to_eta_at_last_update"] > 24 * 90
].copy()

long_eta["days_to_eta_at_last_update"] = (
    long_eta["hours_to_eta_at_last_update"] / 24
).round(1)

display(
    long_eta[
        [
            "mmsi",
            "ship_type",
            "nav_status",
            "sog",
            "destination",
            "last_update",
            "eta_datetime",
            "hours_to_eta_at_last_update",
            "days_to_eta_at_last_update",
        ]
    ]
)


,mmsi,ship_type,nav_status,sog,destination,last_update,eta_datetime,hours_to_eta_at_last_update,days_to_eta_at_last_update
5,247209600,52.0,0.0,0.1,ITSP,2026-04-16 15:31:53,2026-12-10 08:30:00,5704.968611,237.7
55,247278300,52.0,5.0,0.1,IN PORT,2026-04-16 12:38:44,2026-09-21 07:00:00,3786.354444,157.8
109,247351700,69.0,0.0,0.0,AUGUSTA,2026-04-16 15:39:01,2026-10-02 11:30:00,4051.849722,168.8
334,247231200,84.0,0.0,12.3,ESTAR,2026-04-06 15:45:00,2026-08-04 21:00:00,2885.250000,120.2
619,538012016,80.0,0.0,10.3,MALTA,2026-04-09 09:43:54,2026-09-04 11:00:00,3553.268333,148.1
667,247483800,81.0,0.0,13.9,ITTRS,2026-04-09 17:01:56,2026-11-04 19:00:00,5017.967778,209.1
706,242494400,71.0,0.0,14.4,MTMLA,2026-04-09 23:51:34,2026-12-04 04:00:00,5716.140556,238.2
769,229548000,70.0,0.0,16.0,ESVLC,2026-04-10 21:43:06,2026-12-04 22:00:00,5712.281667,238.0
820,247153450,0.0,15.0,9.1,MARE !,2026-04-10 14:04:52,2026-07-18 16:20:00,2378.252222,99.1
1114,247333600,69.0,0.0,0.0,AUGUSTA,2026-04-16 15:39:13,2026-10-03 20:25:00,4084.763056,170.2


<h3 id="Findings">Findings<a class="anchor-link" href="#Findings">¶</a></h3><p>These ETA labels were considered unreliable and excluded before merging.</p>


In [17]:
tracks_final = tracks_valid_eta.loc[
    tracks_valid_eta["hours_to_eta_at_last_update"] <= 24 * 90
].copy()

print(f"Records before filtering: {len(tracks_valid_eta):,}")
print(f"Records removed: {len(tracks_valid_eta) - len(tracks_final):,}")
print(f"Records retained: {len(tracks_final):,}")


Records before filtering: 379
Records removed: 10
Records retained: 369


<h3 id="Merge-valid-ETA-labels-with-AIS-positions">Merge valid ETA labels with AIS positions<a class="anchor-link" href="#Merge-valid-ETA-labels-with-AIS-positions">¶</a></h3><p>The cleaned vessel-level ETA information (track) is merged with the position records using <code>mmsi</code>. An inner join keeps only position records belonging to vessels with a valid ETA label.</p>


In [18]:
track_columns = [
    "mmsi",
    "eta_datetime",
    "destination",
    "ship_type",
    "nav_status"
]

positions_model = positions_clean.merge(
    tracks_final[track_columns],
    on="mmsi",
    how="inner",
    suffixes=("", "_track")
)

print(f"Position records after merge: {len(positions_model):,}")
print(f"Vessels in tracks_final: {tracks_final['mmsi'].nunique():,}")
print(f"Vessels matched after merge: {positions_model['mmsi'].nunique():,}")


Position records after merge: 312,059
Vessels in tracks_final: 369
Vessels matched after merge: 369


<h3 id="Note">Note<a class="anchor-link" href="#Note">¶</a></h3><p>Not included:
callsign: it is a vessel identifier and does not provide predictive information
flag: does not provide predictive information
ship_type_str: it is the text representation of ship_type, so it is redundant
dimensions: could be uselful bas has a huge number of NAs
msg_count:it is metadata describing the track record rather than the vessel's state or movement
last_update: was used during preprocessing to reconstruct and validate ETA labels, but removed afterward becasue it is no longer needed</p>


<h3 id="Construct-the-prediction-target">Construct the prediction target<a class="anchor-link" href="#Construct-the-prediction-target">¶</a></h3><p>The prediction target is defined as the remaining time, in hours, between each AIS position and the vessel's reported ETA. This converts the reported arrival timestamp into a continuous target suitable for regression.</p>
<p>recorded_at = the vessel's current position at a specific time.
eta_datetime = the arrival time reported for that vessel.
remaining_hours = how many hours are left from that position until the reported arrival.</p>


In [19]:
positions_model["remaining_hours"] = (
    positions_model["eta_datetime"]
    - positions_model["recorded_at"]
).dt.total_seconds() / 3600

display(positions_model["remaining_hours"].describe())

print(
    "Negative remaining-time records:",
    (positions_model["remaining_hours"] < 0).sum()
)


count    312059.000000
mean        150.930645
std         191.675253
min           1.253056
25%          56.724167
50%         100.891667
75%         193.091528
max        2194.739444
Name: remaining_hours, dtype: float64


Negative remaining-time records: 0


<h3 id="Note">Note<a class="anchor-link" href="#Note">¶</a></h3><p>No AIS positions that occur after the reported ETA!</p>


<h2 id="Remove-Emojis">Remove Emojis<a class="anchor-link" href="#Remove-Emojis">¶</a></h2>


In [20]:
import re

# Match emojis and related invisible formatting characters
emoji_pattern = re.compile(
    "["
    "\U0001F300-\U0001FAFF"
    "\u2600-\u26FF"
    "\u2700-\u27BF"
    "\uFE0E-\uFE0F"  # Emoji variation selectors
    "\u200D"         # Zero-width joiner
    "]+",
    flags=re.UNICODE,
)

string_columns = positions_model.select_dtypes(
    include=["object", "string"]
).columns

for col in string_columns:
    positions_model[col] = (
        positions_model[col]
        .astype("string")
        .str.replace(emoji_pattern, "", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

print("Emojis and invisible emoji characters removed.")


Emojis and invisible emoji characters removed.


In [21]:
for col in string_columns:
    print(f"\n{col}")
    print(positions_model[col].dropna().unique()[:10])



name
<StringArray>
[       'DANTE A',      'MAC TOKYO', 'MINERVA JOANNA',    'F.LLI FAZIO',
   'MARAN PYTHIA',    'STAR LEGEND',         'SIBLLA', 'GENESIS CEMENT',
       'MED URLA',      'AMBER BEE']
Length: 10, dtype: string

category
<StringArray>
['ship']
Length: 1, dtype: string

nav_status_str
<StringArray>
[             'Alla fonda', 'In navigazione a motore',
  'Manovrabilità limitata',              'Ormeggiata',
   'In navigazione a vela',            'Non definito',
 'Vincolata dal pescaggio']
Length: 7, dtype: string

destination
<StringArray>
[        'ITAUG', 'AUGUSTA-ITALY',       'AUGUSTA',         'ITSIR',
        'IT CTA',    'IT AUGUSTA',        'IT CRV',       'PIRAEUS',
        'IT SPA',        'IT AUG']
Length: 10, dtype: string

manoeuvre_str
<StringArray>
['Non disponibile', 'Manovra speciale in corso', 'Nessuna manovra speciale']
Length: 3, dtype: string

destination_track
<StringArray>
[      'ITPMO', 'BANDIRMA-TR',     'PIRAEUS',     'PALERMO',      'CO CVE',

<h1 id="Feature-Engineering">Feature Engineering<a class="anchor-link" href="#Feature-Engineering">¶</a></h1>


In [22]:
column_summary = pd.DataFrame({
    "Column": positions_model.columns,
    "Data Type": positions_model.dtypes.astype(str),
    "Missing Values": positions_model.isna().sum().values,
    "Missing (%)": (
        positions_model.isna().mean() * 100
    ).round(2).values
})

display(column_summary)

print(f"Total columns: {positions_model.shape[1]}")


,Column,Data Type,Missing Values,Missing (%)
id,id,int64,0,0.00
mmsi,mmsi,int64,0,0.00
name,name,string,39366,12.61
category,category,string,0,0.00
nav_status,nav_status,float64,0,0.00
nav_status_str,nav_status_str,string,0,0.00
sog,sog,float64,0,0.00
cog,cog,float64,9772,3.13
heading,heading,float64,18377,5.89
lat,lat,float64,0,0.00


Total columns: 22


<h2 id="Handle-missing-values">Handle missing values<a class="anchor-link" href="#Handle-missing-values">¶</a></h2>


In [23]:
missing_features = [
    "name",
    "destination",
    "draught",
    "cog",
    "heading"
]

positions_model[missing_features].isna().sum().to_frame(
    name="Missing Values"
)


,Missing Values
name,39366
destination,40732
draught,39366
cog,9772
heading,18377


In [24]:
positions_model["destination"] = positions_model["destination"].fillna("Unknown")

# Fill missing numerical values with the median
numeric_features = ["draught", "cog", "heading"]

for feature in numeric_features:
    positions_model[feature] = positions_model[feature].fillna(
        positions_model[feature].median()
    )

# print
positions_model[["name", "destination", "draught", "cog", "heading"]].isna().sum()


name           39366
destination        0
draught            0
cog                0
heading            0
dtype: int64


<p>Before continuing with feature engineering, I handled the remaining missing values. Since <code>name</code> is only an identifier and won't be used for modelling, I left it unchanged. For <code>destination</code>, I replaced missing values with <code>"Unknown"</code> to preserve those records as a separate category. For the numerical features (<code>draught</code>, <code>cog</code>, and <code>heading</code>), I used the median because it is less sensitive to outliers than the mean.</p>


<h2 id="Temporal-features">Temporal features<a class="anchor-link" href="#Temporal-features">¶</a></h2>


In [25]:
positions_model["hour"] = positions_model["recorded_at"].dt.hour
positions_model["day_of_week"] = positions_model["recorded_at"].dt.dayofweek
positions_model["month"] = positions_model["recorded_at"].dt.month
positions_model["is_weekend"] = (
    positions_model["day_of_week"] >= 5
).astype(int)

positions_model[
    ["recorded_at", "hour", "day_of_week", "month", "is_weekend"]
].head()


,recorded_at,hour,day_of_week,month,is_weekend
0,2026-04-03 10:42:38,10,4,4,0
1,2026-04-03 10:42:38,10,4,4,0
2,2026-04-03 10:42:38,10,4,4,0
3,2026-04-03 10:42:38,10,4,4,0
4,2026-04-03 10:42:38,10,4,4,0


<h2 id="Finding">Finding<a class="anchor-link" href="#Finding">¶</a></h2><p>I created four temporal features (<code>hour</code>, <code>day_of_week</code>, <code>month</code>, and <code>is_weekend</code>) because they might contain useful information that isn't directly captured by the raw timestamp. Later, I'll let the model determine whether these features actually improve prediction performance.</p>


<h2 id="Categorical-Columns">Categorical Columns<a class="anchor-link" href="#Categorical-Columns">¶</a></h2>


In [26]:
categorical_features = [
    "category",
    "nav_status_str",
    "manoeuvre_str",
    "destination"
]

positions_model[categorical_features].nunique().sort_values()


category            1
manoeuvre_str       3
nav_status_str      7
destination       295
dtype: int64


In [27]:
# Remove the constant feature
#positions_model = positions_model.drop(columns="category")

# One-hot encode the low-cardinality categorical features
positions_model = pd.get_dummies(
    positions_model,
    columns=["manoeuvre_str", "nav_status_str"],
    drop_first=True,
    dtype=int
)

print(f"Remaining columns: {positions_model.shape[1]}")

#positions_model.head()


Remaining columns: 32


<h3 id="Finding">Finding<a class="anchor-link" href="#Finding">¶</a></h3><p>I removed the constant <code>category</code> column and converted <code>manoeuvre_str</code> and <code>nav_status_str</code> into numerical dummy variables. This increased the dataset to 31 columns. I left <code>destination</code> unchanged for now because it has many unique values and needs to be inspected separately.</p>


In [28]:
# Inspect the destination names
destination_counts = positions_model["destination"].value_counts()

# Most common destinations
print(destination_counts.head(20))

# Summary statistics
print(destination_counts.describe())

# Inspect possible variations of Augusta
print(
    positions_model[
        positions_model["destination"].str.contains("AUG", case=False, na=False)
    ]["destination"].value_counts()
)


destination
ITAUG               48707
Unknown             40732
IT AUG              32140
AUGUSTA             26873
IT AUGUSTA          14750
ITCTA               11529
MTMLA>ITCTA          7525
ITSIR                5259
SANTA PANAGIA        3720
MALTA                3605
MTMLA                3498
ITSPA FOR ORDERS     3479
MTMAR                3399
FOR ORDERS           3007
IT SPA               2953
ITGIT                2407
ITSAL>ITCTA          2216
ITCTA>ITSAL          2087
TRIST                2048
ITGOA                1911
Name: count, dtype: Int64
count          295.0
mean     1057.827119
std      4536.785946
min              1.0
25%            125.5
50%            275.0
75%            466.0
max          48707.0
Name: count, dtype: Float64
destination
ITAUG              48707
IT AUG             32140
AUGUSTA            26873
IT AUGUSTA         14750
AUGUSTA-ITALY       1533
ITAUG>ITAUG          552
TRDRC>ITAUG          482
ITAUG>EGEDK          371
TRALI>ITAUG          263
AUGUSTA. 

In [29]:
# Standardize common destination names
destination_mapping = {
    # Augusta
    "IT AUG": "ITAUG",
    "IT AUGUSTA": "ITAUG",
    "AUGUSTA": "ITAUG",
    "AUGUSTA-ITALY": "ITAUG",
    "AUGUSTA.": "ITAUG",

    # Naples
    "IT NAP": "ITNAP",

    # Syracuse
    "IT SPA": "ITSPA",
}

positions_model["destination"] = positions_model["destination"].replace(
    destination_mapping
)

# Check the updated destination counts
positions_model["destination"].value_counts().head(20)


destination
ITAUG               124257
Unknown              40732
ITCTA                11529
MTMLA>ITCTA           7525
ITSIR                 5259
ITSPA                 4777
SANTA PANAGIA         3720
MALTA                 3605
MTMLA                 3498
ITSPA FOR ORDERS      3479
MTMAR                 3399
ITNAP                 3389
FOR ORDERS            3007
ITGIT                 2407
ITSAL>ITCTA           2216
ITCTA>ITSAL           2087
TRIST                 2048
ITGOA                 1911
ITPMO                 1522
GIGIB                 1484
Name: count, dtype: Int64


In [30]:
# Group infrequent destinations into "Other"
destination_counts = positions_model["destination"].value_counts()

frequent_destinations = destination_counts[
    destination_counts >= 2000
].index

positions_model["destination"] = positions_model["destination"].where(
    positions_model["destination"].isin(frequent_destinations),
    "Other"
)

# Check the updated distribution
positions_model["destination"].value_counts()


destination
ITAUG               124257
Other                85125
Unknown              40732
ITCTA                11529
MTMLA>ITCTA           7525
ITSIR                 5259
ITSPA                 4777
SANTA PANAGIA         3720
MALTA                 3605
MTMLA                 3498
ITSPA FOR ORDERS      3479
MTMAR                 3399
ITNAP                 3389
FOR ORDERS            3007
ITGIT                 2407
ITSAL>ITCTA           2216
ITCTA>ITSAL           2087
TRIST                 2048
Name: count, dtype: Int64


In [31]:
# One-hot encode destination
positions_model = pd.get_dummies(
    positions_model,
    columns=["destination"],
    drop_first=True,
    dtype=int
)

print(f"Remaining columns: {positions_model.shape[1]}")


Remaining columns: 48


In [32]:
positions_model.columns.tolist()


['id',
 'mmsi',
 'name',
 'category',
 'nav_status',
 'sog',
 'cog',
 'heading',
 'lat',
 'lon',
 'draught',
 'recorded_at',
 'created_at',
 'updated_at',
 'eta_datetime',
 'destination_track',
 'ship_type',
 'nav_status_track',
 'remaining_hours',
 'hour',
 'day_of_week',
 'month',
 'is_weekend',
 'manoeuvre_str_Nessuna manovra speciale',
 'manoeuvre_str_Non disponibile',
 'nav_status_str_In navigazione a motore',
 'nav_status_str_In navigazione a vela',
 'nav_status_str_Manovrabilità limitata',
 'nav_status_str_Non definito',
 'nav_status_str_Ormeggiata',
 'nav_status_str_Vincolata dal pescaggio',
 'destination_ITAUG',
 'destination_ITCTA',
 'destination_ITCTA>ITSAL',
 'destination_ITGIT',
 'destination_ITNAP',
 'destination_ITSAL>ITCTA',
 'destination_ITSIR',
 'destination_ITSPA',
 'destination_ITSPA FOR ORDERS',
 'destination_MALTA',
 'destination_MTMAR',
 'destination_MTMLA',
 'destination_MTMLA>ITCTA',
 'destination_Other',
 'destination_SANTA PANAGIA',
 'destination_TRIST',
 'de

<h3 id="Finding">Finding<a class="anchor-link" href="#Finding">¶</a></h3><p>I standardized the most obvious destination name variations and grouped infrequent destinations into an <code>"Other"</code> category. This simplified the feature by reducing the number of categories while keeping the most common destinations and avoiding an unnecessarily large number of encoded columns.</p>
<p><strong>Note:</strong> I kept <code>"Unknown"</code> as a separate category so I could preserve those observations without making the preprocessing more complicated.</p>


In [33]:
positions_model["ship_type"].value_counts(dropna=False)


ship_type
70.0    77210
80.0    70078
89.0    31509
81.0    25920
71.0    21156
69.0    15576
79.0    13211
90.0    10090
52.0     9009
31.0     8368
73.0     7526
82.0     4690
84.0     3794
0.0      3444
60.0     2953
74.0     2500
54.0     2410
72.0      837
75.0      739
85.0      635
83.0      235
37.0      169
Name: count, dtype: int64


In [34]:
# Treat ship_type as a categorical AIS code
positions_model["ship_type"] = (
    positions_model["ship_type"]
    .fillna(-1)
    .astype(int)
    .astype(str)
)


In [35]:
positions_model = pd.get_dummies(
    positions_model,
    columns=["ship_type"],
    drop_first=True,
    dtype=int
)


In [36]:
[col for col in positions_model.columns if col.startswith("ship_type")]


['ship_type_31',
 'ship_type_37',
 'ship_type_52',
 'ship_type_54',
 'ship_type_60',
 'ship_type_69',
 'ship_type_70',
 'ship_type_71',
 'ship_type_72',
 'ship_type_73',
 'ship_type_74',
 'ship_type_75',
 'ship_type_79',
 'ship_type_80',
 'ship_type_81',
 'ship_type_82',
 'ship_type_83',
 'ship_type_84',
 'ship_type_85',
 'ship_type_89',
 'ship_type_90']


<h2 id="Final-Dataset-Review">Final Dataset Review<a class="anchor-link" href="#Final-Dataset-Review">¶</a></h2>


In [37]:
# Final dataset review

print(f"Rows: {positions_model.shape[0]:,}")
print(f"Columns: {positions_model.shape[1]}")

print("\nData types:")
print(positions_model.dtypes.value_counts())

print("\nRemaining object columns:")
positions_model.select_dtypes(include=["object", "string"]).columns.tolist()

print("\nDuplicate rows:", positions_model.duplicated().sum())

print("\nMissing values:")
missing = positions_model.isna().sum().sort_values(ascending=False)
print(missing[missing > 0])


Rows: 312,059
Columns: 68

Data types:
int64             49
float64            9
datetime64[us]     4
string             3
int32              3
Name: count, dtype: int64

Remaining object columns:

Duplicate rows: 0

Missing values:
name                 39366
destination_track     2042
dtype: int64


<h3 id="Finding">Finding<a class="anchor-link" href="#Finding">¶</a></h3><p>The only remaining missing values are in the <code>name</code> and <code>destination_track</code> columns, which are identifier/raw text fields and will be removed before model training.</p>


<h2 id="Columns-to-drop">Columns to drop<a class="anchor-link" href="#Columns-to-drop">¶</a></h2>


In [38]:
print(positions_model[["nav_status", "nav_status_track"]].head())

print("\nUnique nav_status:", positions_model["nav_status"].unique())
print("\nUnique nav_status_track:", positions_model["nav_status_track"].unique())


   nav_status  nav_status_track
0         1.0               0.0
1         0.0               0.0
2         3.0               0.0
3         3.0               3.0
4         1.0               0.0

Unique nav_status: [ 1.  0.  3.  5.  8. 15.  4.]

Unique nav_status_track: [ 0.  3.  8. 15.]


<p>nav_status is the current status at the time of the AIS message, which is more relevant for predicting ETA.   so we keep it</p>


In [39]:
# Final review before creating the modelling dataset

print("Dataset shape:", positions_model.shape)

print("\nAll columns:")
for col in positions_model.columns:
    print(f"- {col}")

# Columns to remove before saving the modelling dataset
features_to_drop = [
    "id",
    "name",
    "created_at",
    "updated_at",
    "destination_track",
    "nav_status_track"
]

print("\nColumns to drop:")
print(features_to_drop)


Dataset shape: (312059, 68)

All columns:
- id
- mmsi
- name
- category
- nav_status
- sog
- cog
- heading
- lat
- lon
- draught
- recorded_at
- created_at
- updated_at
- eta_datetime
- destination_track
- nav_status_track
- remaining_hours
- hour
- day_of_week
- month
- is_weekend
- manoeuvre_str_Nessuna manovra speciale
- manoeuvre_str_Non disponibile
- nav_status_str_In navigazione a motore
- nav_status_str_In navigazione a vela
- nav_status_str_Manovrabilità limitata
- nav_status_str_Non definito
- nav_status_str_Ormeggiata
- nav_status_str_Vincolata dal pescaggio
- destination_ITAUG
- destination_ITCTA
- destination_ITCTA>ITSAL
- destination_ITGIT
- destination_ITNAP
- destination_ITSAL>ITCTA
- destination_ITSIR
- destination_ITSPA
- destination_ITSPA FOR ORDERS
- destination_MALTA
- destination_MTMAR
- destination_MTMLA
- destination_MTMLA>ITCTA
- destination_Other
- destination_SANTA PANAGIA
- destination_TRIST
- destination_Unknown
- ship_type_31
- ship_type_37
- ship_type_52
-

<h2 id="Save-dataset-with-meta-data">Save dataset with meta data<a class="anchor-link" href="#Save-dataset-with-meta-data">¶</a></h2><p>the following coumns were kepy becasue they are iportant for converting predicted remaining hours back into predicted ETAs during the discrepancy analysis.</p>


In [40]:
# Keep metadata for different evaluation strategies and discrepancy analysis
metadata_columns = ["mmsi", "recorded_at", "eta_datetime"]

# Remove unnecessary columns while keeping metadata
columns_to_drop = features_to_drop.copy()

# Remove constant feature
columns_to_drop.append("category")

model_data_with_metadata = positions_model.drop(
    columns=columns_to_drop
).copy()

model_data_with_metadata.to_csv(
    "../data/processed/model_data_with_metadata.csv",
    index=False
)

print("Saved shape:", model_data_with_metadata.shape)


Saved shape: (312059, 61)


In [41]:
# Final quality check of the modelling dataset

print("model_data_with_metadata") 


print(f"\nDataset shape: {model_data_with_metadata.shape}")

print("\nData types:")
print(model_data_with_metadata.dtypes.value_counts())

print("\nMissing values:")
missing = model_data_with_metadata.isna().sum()
missing = missing[missing > 0]

if missing.empty:
    print("No missing values found.")
else:
    print(missing)

print("\nDuplicate rows:")
duplicates = model_data_with_metadata.duplicated().sum()

if duplicates == 0:
    print("No duplicate rows found.")
else:
    print(f"{duplicates} duplicate rows found.")

print("\nRemaining non-numeric columns:")
non_numeric = model_data_with_metadata.select_dtypes(exclude=["number"]).columns.tolist()

if len(non_numeric) == 0:
    print("All remaining features are numeric.")
else:
    print(non_numeric)


model_data_with_metadata

Dataset shape: (312059, 61)

Data types:
int64             48
float64            8
int32              3
datetime64[us]     2
Name: count, dtype: int64

Missing values:
No missing values found.

Duplicate rows:
No duplicate rows found.

Remaining non-numeric columns:
['recorded_at', 'eta_datetime']
